# Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import yaml

# >>> EDIT THIS to your dataset folder name in Drive <<<
ROOT = Path("/content/drive/MyDrive/IsaacSim_Dataset_Simple")  # change if needed
# Example: ROOT = Path("/content/drive/MyDrive/IsaacSim_Dataset_Sim")

classes_txt = ROOT / "classes.txt"
assert ROOT.exists(), f"ROOT not found: {ROOT}"

# Load class names
assert classes_txt.exists(), f"Missing classes.txt at: {classes_txt}"
names_list = [l.strip() for l in classes_txt.read_text().splitlines() if l.strip()]
names = {i: n for i, n in enumerate(names_list)}
print(f"Loaded {len(names)} classes.")

# Detect dataset layout
# Layout A (most common): ROOT/images/{train,val,test} and ROOT/labels/{train,val,test}
layoutA = (ROOT / "images" / "train").exists() and (ROOT / "labels" / "train").exists()

# Layout B: ROOT/{train,val,test}/images and ROOT/{train,val,test}/labels
layoutB = (ROOT / "train" / "images").exists() and (ROOT / "train" / "labels").exists()

assert layoutA or layoutB, (
    "Couldn't detect splits. Expected either:\n"
    "A) ROOT/images/train + ROOT/labels/train\n"
    "or\n"
    "B) ROOT/train/images + ROOT/train/labels"
)

if layoutA:
    train_path = "images/train"
    val_path   = "images/val"
    test_path  = "images/test"
    print("Detected Layout A: ROOT/images/{train,val,test} and ROOT/labels/{train,val,test}")
else:
    train_path = "train/images"
    val_path   = "val/images"
    test_path  = "test/images"
    print("Detected Layout B: ROOT/{train,val,test}/images and ROOT/{train,val,test}/labels")

data = {
    "path": str(ROOT),
    "train": train_path,
    "val": val_path,
    "test": test_path,
    "names": names
}

OUT_YAML = Path("/content/isaac_correct.yaml")
OUT_YAML.write_text(yaml.safe_dump(data, sort_keys=False))
print("\nWrote:", OUT_YAML)
print(OUT_YAML.read_text())


In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO

DATA_YAML = "/content/isaac_correct.yaml"

model = YOLO("yolov8n.pt")
train_res = model.train(
    data=DATA_YAML,
    imgsz=640,
    epochs=100,
    batch=16,
    patience=80,
    device=0,
    name="isaac_clean_split"
)

best_pt = f"{train_res.save_dir}/weights/best.pt"
print("Best checkpoint:", best_pt)


In [ ]:
from ultralytics import YOLO

DATA_YAML = "/content/isaac_correct.yaml"
best_pt = f"{train_res.save_dir}/weights/best.pt"

m = YOLO(best_pt)

# Clean test evaluation (this is your baseline)
test_res = m.val(data=DATA_YAML, split="test", imgsz=640, conf=0.25, iou=0.7)


In [ ]:
from pathlib import Path
import shutil

src = Path(best_pt)
dst = Path("/content/drive/MyDrive/models/isaac_clean_split_best.pt")
dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(src, dst)
print("Saved model to:", dst)


In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO

In [ ]:
from ultralytics import YOLO

m = YOLO(best_pt)
m.predict(
    source="/content/drive/MyDrive/IsaacSim_Dataset_Simple/images/test",
    imgsz=640,
    conf=0.25,
    save=True
)
print("Saved predictions to /content/runs/detect/predict/")


##Create the Learnable Patch

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patch_size = (80, 80)

# Learnable patch initialized with random values
patch = torch.rand(3, *patch_size, device=device, requires_grad=True)

# Setup optimizer
optimizer = torch.optim.Adam([patch], lr=0.05)

print(f"Patch shape: {patch.shape} | Device: {device}")


## Apply Appearance Transformations

In [ ]:
import torchvision.transforms as T

appearance_transforms = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomPerspective(distortion_scale=0.5, p=0.5),
    T.GaussianBlur(3),
])

def apply_appearance_transforms(patch):
    return appearance_transforms(patch)



## EOT — Spatial + Projective Transformations

In [ ]:
!pip -q install kornia

import kornia

def apply_eot(patch):
    # Simulate different angles/scales
    affine = kornia.augmentation.RandomAffine(degrees=20, translate=0.1, scale=(0.8, 1.2))
    return affine(patch.unsqueeze(0)).squeeze(0)


## Add Patch to Image

In [ ]:
import torch
import random

def overlay_patch(image, patch, x_offset=None, y_offset=None):
    patched = image.clone()
    _, H_img, W_img = image.shape
    _, H_patch, W_patch = patch.shape

    # Calculate valid bounds
    max_y = H_img - H_patch
    max_x = W_img - W_patch

    if max_y < 0 or max_x < 0:
        raise ValueError("Patch is larger than the image.")

    # If no offsets are given, random position
    if x_offset is None:
        x_offset = random.randint(0, max_x)
    else:
        x_offset = min(max(x_offset, 0), max_x)

    if y_offset is None:
        y_offset = random.randint(0, max_y)
    else:
        y_offset = min(max(y_offset, 0), max_y)

    # Apply the patch
    patched[:, y_offset:y_offset + H_patch, x_offset:x_offset + W_patch] = patch
    return patched



# Load trained YOLOv8 model


In [ ]:
YOLO_MODEL_PATH = "/content/drive/MyDrive/models/isaac_clean_split_best.pt"  # <- your trained model


In [ ]:
from ultralytics import YOLO

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load trained YOLOv8 model
model = YOLO(YOLO_MODEL_PATH)
model.fuse()

model.model.to(device)
model.to(device)


In [ ]:
def overlay_patch(image, patch, x_offset=0, y_offset=0):
    patched_image = image.clone()
    _, H, W = image.shape
    ph, pw = patch.shape[1:]

    # Ensure patch fits
    if y_offset + ph > H or x_offset + pw > W:
        raise ValueError("Patch placement is outside image bounds.")

    patched_image[:, y_offset:y_offset+ph, x_offset:x_offset+pw] = patch
    return patched_image


# Optimizing Patch on a folder (e.g. billboard01)

In [ ]:
from torch.utils.data import DataLoader
import torchvision.transforms as T
import os
from glob import glob


# Define image + label paths
image_dir = "/content/drive/MyDrive/IsaacSim_Dataset_Simple/images/test"
label_dir = "/content/drive/MyDrive/IsaacSim_Dataset_Simple/labels/test"



In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms

# Load all image paths from the directory
image_paths = [os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith(".jpg") or fname.endswith(".png")]

Patch optimized over multiple random scales to improve robustness to viewpoint distance.

In [ ]:
# =========================
# 3-seed patch optimization + 5-scale robustness (ALL 5 scales per image per step)
# Uses a SINGLE backward per image (sum of 5-scale losses) to avoid "backward twice" error.
# Saves: bill1a.pt, bill1b.pt, bill1c.pt and best_bill1.pt
# =========================

import os
import random
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

# ---------- Config ----------
SEEDS = [0, 1, 2]
SEED_TAGS = {0: "a", 1: "b", 2: "c"}   # bill1a/b/c naming

SAVE_DIR = "./patch_outputs"
# SAVE_DIR = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

K = 400
TH = 0.25
LAMBDA_TH = 1.0
LR = 0.03
STEPS = 200

# Anchor position for BASE patch size
X_OFFSET = 300
Y_OFFSET = 300

# 5 scales (zoom in/out robustness)
SCALES = [0.85, 1.00, 1.20, 1.45, 1.80]

# Optional: limit images per step to reduce GPU load (None = all)
# If GPU crashes, set to 8 or 16.
IMAGES_PER_STEP = None  # e.g. 16

# ---------- Setup ----------
device = next(model.model.parameters()).device
print("Model device:", device)

nc = int(getattr(model.model, "nc", 80))
print("Model nc:", nc)

# Recommended stability
model.model.eval()
image_paths = sorted(image_paths)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(False)

def get_any_confidence(preds, nc: int) -> torch.Tensor:
    """
    Handles YOLOv8 outputs that may be (N,D) or (D,N) or (1,N,D).
    D = 4 + nc OR 5 + nc.
    Returns conf_any: (N,) = max class score per box (with obj if present).
    """
    if preds.dim() == 3:
        preds = preds[0]

    # If transposed (D,N), convert to (N,D)
    if preds.shape[0] in (4 + nc, 5 + nc) and preds.shape[1] not in (4 + nc, 5 + nc):
        preds = preds.t()

    D = preds.shape[1]
    if D == 5 + nc:
        obj = preds[:, 4]
        cls_scores = preds[:, 5:]
        return obj * cls_scores.max(dim=1).values

    if D == 4 + nc:
        cls_scores = preds[:, 4:]
        return cls_scores.max(dim=1).values

    raise ValueError(f"Unexpected preds shape {preds.shape}. Expected D={5+nc} or D={4+nc}.")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((640, 640)),
])

# Patch base must already exist in your notebook/script
patch_base = patch.to(device)
_, BASE_H, BASE_W = patch_base.shape

def resize_patch(p: torch.Tensor, scale: float) -> torch.Tensor:
    """Resize patch tensor (C,H,W) by scale using torch ops (keeps grads)."""
    C, H, W = p.shape
    new_h = max(1, int(round(H * scale)))
    new_w = max(1, int(round(W * scale)))
    return F.interpolate(
        p.unsqueeze(0),
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

def center_preserving_offsets(x_offset: int, y_offset: int, new_h: int, new_w: int):
    """Adjust top-left offsets so patch stays roughly centered when size changes."""
    x = int(round(x_offset - (new_w - BASE_W) / 2))
    y = int(round(y_offset - (new_h - BASE_H) / 2))
    return x, y

# Store results for best selection
seed_results = {}  # seed -> dict(best_loss, best_patch_cpu, save_path)

for seed in SEEDS:
    tag = SEED_TAGS[seed]
    save_path = os.path.join(SAVE_DIR, f"bill1{tag}.pt")

    print("\n" + "=" * 60)
    print(f"Running seed = {seed}  -> will save: {save_path}")
    print("=" * 60)

    set_seed(seed)

    # fresh patch + optimizer per seed
    patch4 = torch.rand_like(patch_base, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([patch4], lr=LR)

    best_loss = float("inf")
    best_patch4_cpu = patch4.detach().clone().cpu()

    for step in range(STEPS):
        optimizer.zero_grad(set_to_none=True)

        # Optionally sample subset images (speed/stability)
        if IMAGES_PER_STEP is None or IMAGES_PER_STEP >= len(image_paths):
            step_paths = image_paths
        else:
            step_paths = random.sample(image_paths, k=IMAGES_PER_STEP)

        # We do ALL 5 scales per image
        denom = len(step_paths) * len(SCALES)

        total_loss_val = 0.0
        total_boxes_over_th = 0

        for img_path in step_paths:
            img = Image.open(img_path).convert("RGB")
            image_tensor = transform(img).to(device)

            # Sum 5-scale losses, then backward ONCE per image
            loss_img = 0.0

            for sc in SCALES:
                # IMPORTANT: build fresh patch transforms each scale (avoids "backward twice" on same graph)
                t_patch = apply_appearance_transforms(patch4)
                t_patch = apply_eot(t_patch)

                # scale patch
                s_patch = resize_patch(t_patch, sc).clamp(0, 1)
                _, ph, pw = s_patch.shape
                x_off, y_off = center_preserving_offsets(X_OFFSET, Y_OFFSET, ph, pw)

                # overlay
                patched_image = overlay_patch(
                    image_tensor, s_patch, x_offset=x_off, y_offset=y_off
                )
                patched_input = patched_image.unsqueeze(0)

                # Forward raw head (your working call)
                preds = model.model(patched_input)[0]

                # conf
                conf_any = get_any_confidence(preds, nc)

                # losses
                topk = torch.topk(conf_any, k=min(K, conf_any.numel())).values
                loss_topk = -torch.log(topk + 1e-6).sum()
                loss_th = -torch.relu(conf_any - TH).sum()
                loss = loss_topk + LAMBDA_TH * loss_th

                loss_img = loss_img + loss

                # logging (detach only)
                total_loss_val += loss.detach().item()
                total_boxes_over_th += (conf_any > TH).sum().item()

            # single backward for this image across all 5 scales
            (loss_img / max(1, denom)).backward()

        optimizer.step()
        patch4.data.clamp_(0, 1)

        avg_loss = total_loss_val / max(1, denom)
        avg_boxes_over_th = total_boxes_over_th / max(1, denom)

        # Track best patch by lowest avg_loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_patch4_cpu = patch4.detach().clone().cpu()

        if step % 10 == 0:
            print(
                f"[seed {seed}] Step {step:3d} | Loss: {avg_loss:.4f} | "
                f"Avg boxes>TH({TH}): {avg_boxes_over_th:.2f} | Scales: {SCALES}"
            )

    # Save the best patch for this seed
    torch.save(best_patch4_cpu, save_path)
    print(f"Saved seed {seed} best patch to: {save_path}")

    seed_results[seed] = {
        "best_loss": best_loss,
        "best_patch_cpu": best_patch4_cpu,
        "save_path": save_path,
    }

# Choose overall best patch by lowest best_loss
best_seed = min(seed_results.keys(), key=lambda s: seed_results[s]["best_loss"])
best_patch_overall = seed_results[best_seed]["best_patch_cpu"]
best_path = os.path.join(SAVE_DIR, "best_bill1.pt")
torch.save(best_patch_overall, best_path)

print("\n" + "#" * 70)
print(f"Best seed = {best_seed}  with best_loss = {seed_results[best_seed]['best_loss']:.6f}")
print(f"Saved overall best patch to: {best_path}")
print("#" * 70)


In [ ]:
import os
import torch
import torchvision.utils as vutils

# ----------------------------
# Config
# ----------------------------
# Where patches were saved by training (local runtime)
SRC_DIR = "./patch_outputs"

# Where to store on Google Drive (persistent)
OUT_DIR = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs_saved"
os.makedirs(OUT_DIR, exist_ok=True)

# Desired names on Drive
name_map = {
    "isaac_patch_seed0": "bill1a.pt",
    "isaac_patch_seed1": "bill1b.pt",
    "isaac_patch_seed2": "bill1c.pt",
    "isaac_patch_best":  "best_bill1.pt",
}

# ----------------------------
# Load + Save (png + pt) to Drive
# ----------------------------
for out_name, src_file in name_map.items():
    src_path = os.path.join(SRC_DIR, src_file)
    if not os.path.exists(src_path):
        print(f"❌ Missing: {src_path} (skip)")
        continue

    p = torch.load(src_path, map_location="cpu").detach().clone().clamp(0, 1).cpu()

    png_path = os.path.join(OUT_DIR, f"{out_name}.png")
    pt_path  = os.path.join(OUT_DIR, f"{out_name}.pt")

    vutils.save_image(p, png_path)
    torch.save(p, pt_path)

    print(f"✅ Saved: {png_path}")
    print(f"✅ Saved: {pt_path}")

# ----------------------------
# Optional: simple alias for demos (same as best)
# ----------------------------
best_src = os.path.join(SRC_DIR, "best_bill1.pt")
if os.path.exists(best_src):
    best_patch = torch.load(best_src, map_location="cpu").detach().clone().clamp(0, 1).cpu()

    vutils.save_image(best_patch, os.path.join(OUT_DIR, "isaac_patch.png"))
    torch.save(best_patch, os.path.join(OUT_DIR, "isaac_patch.pt"))

    print("⭐ Saved aliases: isaac_patch.png and isaac_patch.pt (== isaac_patch_best)")
else:
    print(f"❌ Missing best patch: {best_src}")

print("\nDone. Output folder:", OUT_DIR)


In [ ]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision import transforms

# ----------------------------
# Config
# ----------------------------
PATCH_PT = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs_saved/isaac_patch_best.pt"
# or: PATCH_PT = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs_saved/isaac_patch.pt"

OUT_DIR = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_eval_sizes"
os.makedirs(OUT_DIR, exist_ok=True)

DEMO_IMAGE_PATH = image_paths[10]   # change if you want

# anchor offsets for BASE patch size
X_OFFSET = 300
Y_OFFSET = 300

# your requested sizes
SCALES = [0.85, 1.00, 1.20, 1.45, 1.80]

TH = 0.25  # evaluation threshold (same as training)

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((640, 640)),
])

# ----------------------------
# Helpers
# ----------------------------
def get_any_confidence(preds, nc: int) -> torch.Tensor:
    """Works with YOLOv8-ish head outputs (N,D) or (D,N) or (1,N,D)."""
    if preds.dim() == 3:
        preds = preds[0]

    # If transposed (D,N) -> (N,D)
    if preds.shape[0] in (4 + nc, 5 + nc) and preds.shape[1] not in (4 + nc, 5 + nc):
        preds = preds.t()

    D = preds.shape[1]
    if D == 5 + nc:
        obj = preds[:, 4]
        cls_scores = preds[:, 5:]
        return obj * cls_scores.max(dim=1).values
    if D == 4 + nc:
        cls_scores = preds[:, 4:]
        return cls_scores.max(dim=1).values

    raise ValueError(f"Unexpected preds shape {preds.shape}. Expected D={5+nc} or D={4+nc}.")

def resize_patch(p: torch.Tensor, scale: float) -> torch.Tensor:
    """Resize patch (C,H,W) by scale using torch ops."""
    C, H, W = p.shape
    new_h = max(1, int(round(H * scale)))
    new_w = max(1, int(round(W * scale)))
    return F.interpolate(
        p.unsqueeze(0),
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

def center_preserving_offsets(x_offset: int, y_offset: int, base_h: int, base_w: int, new_h: int, new_w: int):
    """Keep patch centered around the same place when size changes."""
    x = int(round(x_offset - (new_w - base_w) / 2))
    y = int(round(y_offset - (new_h - base_h) / 2))
    return x, y

def safe_det_count(result0):
    """Try to count detections from Ultralytics Results (if present)."""
    try:
        if hasattr(result0, "boxes") and result0.boxes is not None:
            return int(result0.boxes.shape[0]) if hasattr(result0.boxes, "shape") else int(len(result0.boxes))
    except Exception:
        pass
    return None

# ----------------------------
# Load model info + patch + image
# ----------------------------
assert os.path.exists(PATCH_PT), f"Patch not found: {PATCH_PT}"
assert os.path.exists(DEMO_IMAGE_PATH), f"Image not found: {DEMO_IMAGE_PATH}"

device = next(model.model.parameters()).device
nc = int(getattr(model.model, "nc", 80))

patch_cpu = torch.load(PATCH_PT, map_location="cpu").clamp(0, 1)
base_c, base_h, base_w = patch_cpu.shape

img = Image.open(DEMO_IMAGE_PATH).convert("RGB")
image_tensor = demo_transform(img).to(device)

print("Device:", device)
print("nc:", nc)
print("Patch:", PATCH_PT, "| patch:", tuple(patch_cpu.shape))
print("Demo:", DEMO_IMAGE_PATH, "| image_tensor:", tuple(image_tensor.shape))
print("OUT_DIR:", OUT_DIR)

# ----------------------------
# Clean (reference)
# ----------------------------
with torch.no_grad():
    clean_np = image_tensor.clamp(0, 1).detach().cpu().permute(1, 2, 0).numpy()

    # YOLO plotted detections (for visualization)
    model.model.eval()
    clean_det = model(image_tensor.unsqueeze(0), verbose=False)
    clean_plot_bgr = clean_det[0].plot()
    clean_plot_rgb = cv2.cvtColor(clean_plot_bgr, cv2.COLOR_BGR2RGB)

    # raw-head evaluation (boxes>TH)
    preds_clean = model.model(image_tensor.unsqueeze(0))[0]
    conf_clean = get_any_confidence(preds_clean, nc)
    clean_boxes_over_th = int((conf_clean > TH).sum().item())

    clean_det_count = safe_det_count(clean_det[0])

# Save clean images
clean_nodet_path = os.path.join(OUT_DIR, "clean_nodet.png")
clean_withdet_path = os.path.join(OUT_DIR, "clean_withdet.png")
Image.fromarray((clean_np * 255).astype("uint8")).save(clean_nodet_path)
Image.fromarray(clean_plot_rgb).save(clean_withdet_path)

print(f"Saved clean: {clean_nodet_path}")
print(f"Saved clean: {clean_withdet_path}")

# ----------------------------
# Evaluate each scale
# ----------------------------
rows = []
patched_nodet_imgs = []
patched_withdet_imgs = []

with torch.no_grad():
    patch_gpu_base = patch_cpu.to(device)

    for sc in SCALES:
        # scale patch
        p_sc = resize_patch(patch_gpu_base, sc).clamp(0, 1)
        _, ph, pw = p_sc.shape

        # center-preserving offsets
        x_off, y_off = center_preserving_offsets(X_OFFSET, Y_OFFSET, base_h, base_w, ph, pw)

        # overlay
        patched = overlay_patch(
            image_tensor.clone(),
            p_sc,
            x_offset=x_off,
            y_offset=y_off
        )

        # raw-head metric (boxes > TH)
        preds = model.model(patched.unsqueeze(0))[0]
        conf_any = get_any_confidence(preds, nc)
        boxes_over_th = int((conf_any > TH).sum().item())
        top_conf = float(conf_any.max().item()) if conf_any.numel() > 0 else 0.0

        # YOLO plotted detections
        det = model(patched.unsqueeze(0), verbose=False)
        det_count = safe_det_count(det[0])

        plot_bgr = det[0].plot()
        plot_rgb = cv2.cvtColor(plot_bgr, cv2.COLOR_BGR2RGB)

        # save images
        patched_np = patched.clamp(0, 1).detach().cpu().permute(1, 2, 0).numpy()

        nodet_path = os.path.join(OUT_DIR, f"patched_scale_{sc:.2f}_nodet.png")
        withdet_path = os.path.join(OUT_DIR, f"patched_scale_{sc:.2f}_withdet.png")
        Image.fromarray((patched_np * 255).astype("uint8")).save(nodet_path)
        Image.fromarray(plot_rgb).save(withdet_path)

        patched_nodet_imgs.append(Image.open(nodet_path).convert("RGB"))
        patched_withdet_imgs.append(Image.open(withdet_path).convert("RGB"))

        rows.append({
            "scale": sc,
            "patch_hw": (ph, pw),
            "offset_xy": (x_off, y_off),
            "raw_boxes>TH": boxes_over_th,
            "raw_max_conf": round(top_conf, 4),
            "yolo_det_count": det_count,
            "saved_nodet": nodet_path,
            "saved_withdet": withdet_path
        })

# ----------------------------
# Print evaluation table
# ----------------------------
print("\n=== CLEAN reference ===")
print(f"raw_boxes>TH({TH}) = {clean_boxes_over_th} | yolo_det_count = {clean_det_count}")

print("\n=== PATCHED by scale ===")
for r in rows:
    print(
        f"scale={r['scale']:.2f} | patch={r['patch_hw']} | offset={r['offset_xy']} | "
        f"raw_boxes>TH={r['raw_boxes>TH']} | raw_max_conf={r['raw_max_conf']} | yolo_det_count={r['yolo_det_count']}"
    )

# ----------------------------
# Visualize: clean + all patched
# ----------------------------
# Row 1: no det (clean + 5 scales)
plt.figure(figsize=(20, 7))
plt.subplot(2, 6, 1)
plt.imshow(clean_np)
plt.title("Clean (no det)")
plt.axis("off")

for i, sc in enumerate(SCALES, start=2):
    plt.subplot(2, 6, i)
    plt.imshow(patched_nodet_imgs[i-2])
    plt.title(f"Scale {SCALES[i-2]:.2f} (no det)")
    plt.axis("off")

# Row 2: with det (clean + 5 scales)
plt.subplot(2, 6, 7)
plt.imshow(clean_plot_rgb)
plt.title("Clean (with det)")
plt.axis("off")

for i, sc in enumerate(SCALES, start=8):
    plt.subplot(2, 6, i)
    plt.imshow(patched_withdet_imgs[i-8])
    plt.title(f"Scale {SCALES[i-8]:.2f} (with det)")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision import transforms

# ----------------------------
# Config
# ----------------------------
PATCH_PT = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs_saved/isaac_patch_best.pt"
# or: PATCH_PT = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs_saved/isaac_patch.pt"

OUT_DIR = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_eval_sizes"
os.makedirs(OUT_DIR, exist_ok=True)

DEMO_IMAGE_PATH = image_paths[10]   # change if you want

# anchor offsets for BASE patch size
X_OFFSET = 300
Y_OFFSET = 300

# your requested sizes
SCALES = [0.7, 1.30, 2.00, 1.00, 3.20]

TH = 0.25  # evaluation threshold (same as training)

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((640, 640)),
])

# ----------------------------
# Helpers
# ----------------------------
def get_any_confidence(preds, nc: int) -> torch.Tensor:
    """Works with YOLOv8-ish head outputs (N,D) or (D,N) or (1,N,D)."""
    if preds.dim() == 3:
        preds = preds[0]

    # If transposed (D,N) -> (N,D)
    if preds.shape[0] in (4 + nc, 5 + nc) and preds.shape[1] not in (4 + nc, 5 + nc):
        preds = preds.t()

    D = preds.shape[1]
    if D == 5 + nc:
        obj = preds[:, 4]
        cls_scores = preds[:, 5:]
        return obj * cls_scores.max(dim=1).values
    if D == 4 + nc:
        cls_scores = preds[:, 4:]
        return cls_scores.max(dim=1).values

    raise ValueError(f"Unexpected preds shape {preds.shape}. Expected D={5+nc} or D={4+nc}.")

def resize_patch(p: torch.Tensor, scale: float) -> torch.Tensor:
    """Resize patch (C,H,W) by scale using torch ops."""
    C, H, W = p.shape
    new_h = max(1, int(round(H * scale)))
    new_w = max(1, int(round(W * scale)))
    return F.interpolate(
        p.unsqueeze(0),
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

def center_preserving_offsets(x_offset: int, y_offset: int, base_h: int, base_w: int, new_h: int, new_w: int):
    """Keep patch centered around the same place when size changes."""
    x = int(round(x_offset - (new_w - base_w) / 2))
    y = int(round(y_offset - (new_h - base_h) / 2))
    return x, y

def safe_det_count(result0):
    """Try to count detections from Ultralytics Results (if present)."""
    try:
        if hasattr(result0, "boxes") and result0.boxes is not None:
            return int(result0.boxes.shape[0]) if hasattr(result0.boxes, "shape") else int(len(result0.boxes))
    except Exception:
        pass
    return None

# ----------------------------
# Load model info + patch + image
# ----------------------------
assert os.path.exists(PATCH_PT), f"Patch not found: {PATCH_PT}"
assert os.path.exists(DEMO_IMAGE_PATH), f"Image not found: {DEMO_IMAGE_PATH}"

device = next(model.model.parameters()).device
nc = int(getattr(model.model, "nc", 80))

patch_cpu = torch.load(PATCH_PT, map_location="cpu").clamp(0, 1)
base_c, base_h, base_w = patch_cpu.shape

img = Image.open(DEMO_IMAGE_PATH).convert("RGB")
image_tensor = demo_transform(img).to(device)

print("Device:", device)
print("nc:", nc)
print("Patch:", PATCH_PT, "| patch:", tuple(patch_cpu.shape))
print("Demo:", DEMO_IMAGE_PATH, "| image_tensor:", tuple(image_tensor.shape))
print("OUT_DIR:", OUT_DIR)

# ----------------------------
# Clean (reference)
# ----------------------------
with torch.no_grad():
    clean_np = image_tensor.clamp(0, 1).detach().cpu().permute(1, 2, 0).numpy()

    # YOLO plotted detections (for visualization)
    model.model.eval()
    clean_det = model(image_tensor.unsqueeze(0), verbose=False)
    clean_plot_bgr = clean_det[0].plot()
    clean_plot_rgb = cv2.cvtColor(clean_plot_bgr, cv2.COLOR_BGR2RGB)

    # raw-head evaluation (boxes>TH)
    preds_clean = model.model(image_tensor.unsqueeze(0))[0]
    conf_clean = get_any_confidence(preds_clean, nc)
    clean_boxes_over_th = int((conf_clean > TH).sum().item())

    clean_det_count = safe_det_count(clean_det[0])

# Save clean images
clean_nodet_path = os.path.join(OUT_DIR, "clean_nodet.png")
clean_withdet_path = os.path.join(OUT_DIR, "clean_withdet.png")
Image.fromarray((clean_np * 255).astype("uint8")).save(clean_nodet_path)
Image.fromarray(clean_plot_rgb).save(clean_withdet_path)

print(f"Saved clean: {clean_nodet_path}")
print(f"Saved clean: {clean_withdet_path}")

# ----------------------------
# Evaluate each scale
# ----------------------------
rows = []
patched_nodet_imgs = []
patched_withdet_imgs = []

with torch.no_grad():
    patch_gpu_base = patch_cpu.to(device)

    for sc in SCALES:
        # scale patch
        p_sc = resize_patch(patch_gpu_base, sc).clamp(0, 1)
        _, ph, pw = p_sc.shape

        # center-preserving offsets
        x_off, y_off = center_preserving_offsets(X_OFFSET, Y_OFFSET, base_h, base_w, ph, pw)

        # overlay
        patched = overlay_patch(
            image_tensor.clone(),
            p_sc,
            x_offset=x_off,
            y_offset=y_off
        )

        # raw-head metric (boxes > TH)
        preds = model.model(patched.unsqueeze(0))[0]
        conf_any = get_any_confidence(preds, nc)
        boxes_over_th = int((conf_any > TH).sum().item())
        top_conf = float(conf_any.max().item()) if conf_any.numel() > 0 else 0.0

        # YOLO plotted detections
        det = model(patched.unsqueeze(0), verbose=False)
        det_count = safe_det_count(det[0])

        plot_bgr = det[0].plot()
        plot_rgb = cv2.cvtColor(plot_bgr, cv2.COLOR_BGR2RGB)

        # save images
        patched_np = patched.clamp(0, 1).detach().cpu().permute(1, 2, 0).numpy()

        nodet_path = os.path.join(OUT_DIR, f"patched_scale_{sc:.2f}_nodet.png")
        withdet_path = os.path.join(OUT_DIR, f"patched_scale_{sc:.2f}_withdet.png")
        Image.fromarray((patched_np * 255).astype("uint8")).save(nodet_path)
        Image.fromarray(plot_rgb).save(withdet_path)

        patched_nodet_imgs.append(Image.open(nodet_path).convert("RGB"))
        patched_withdet_imgs.append(Image.open(withdet_path).convert("RGB"))

        rows.append({
            "scale": sc,
            "patch_hw": (ph, pw),
            "offset_xy": (x_off, y_off),
            "raw_boxes>TH": boxes_over_th,
            "raw_max_conf": round(top_conf, 4),
            "yolo_det_count": det_count,
            "saved_nodet": nodet_path,
            "saved_withdet": withdet_path
        })

# ----------------------------
# Print evaluation table
# ----------------------------
print("\n=== CLEAN reference ===")
print(f"raw_boxes>TH({TH}) = {clean_boxes_over_th} | yolo_det_count = {clean_det_count}")

print("\n=== PATCHED by scale ===")
for r in rows:
    print(
        f"scale={r['scale']:.2f} | patch={r['patch_hw']} | offset={r['offset_xy']} | "
        f"raw_boxes>TH={r['raw_boxes>TH']} | raw_max_conf={r['raw_max_conf']} | yolo_det_count={r['yolo_det_count']}"
    )

# ----------------------------
# Visualize: clean + all patched
# ----------------------------
# Row 1: no det (clean + 5 scales)
plt.figure(figsize=(20, 7))
plt.subplot(2, 6, 1)
plt.imshow(clean_np)
plt.title("Clean (no det)")
plt.axis("off")

for i, sc in enumerate(SCALES, start=2):
    plt.subplot(2, 6, i)
    plt.imshow(patched_nodet_imgs[i-2])
    plt.title(f"Scale {SCALES[i-2]:.2f} (no det)")
    plt.axis("off")

# Row 2: with det (clean + 5 scales)
plt.subplot(2, 6, 7)
plt.imshow(clean_plot_rgb)
plt.title("Clean (with det)")
plt.axis("off")

for i, sc in enumerate(SCALES, start=8):
    plt.subplot(2, 6, i)
    plt.imshow(patched_withdet_imgs[i-8])
    plt.title(f"Scale {SCALES[i-8]:.2f} (with det)")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os, random, glob
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# ----------------------------
# Mount Drive if needed
# ----------------------------
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# ----------------------------
# Paths
# ----------------------------
IMAGE_DIR = "/content/drive/MyDrive/Patch_test1/images"
OUT_DIR   = "/content/drive/MyDrive/Patch_test1/preds_10"
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(IMAGE_DIR), f"Folder not found: {IMAGE_DIR}"

# ----------------------------
# Collect images + sample 10
# ----------------------------
exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp")
image_paths = []
for e in exts:
    image_paths += glob.glob(os.path.join(IMAGE_DIR, e))

image_paths = sorted(image_paths)
assert len(image_paths) > 0, f"No images found in: {IMAGE_DIR}"

sample_paths = random.sample(image_paths, k=min(10, len(image_paths)))
print(f"Found {len(image_paths)} images. Sampling {len(sample_paths)}.\n")

# ----------------------------
# Run predictions (batch on paths)
# ----------------------------
model.model.eval()
results = model(sample_paths, verbose=False)  # Ultralytics accepts list of file paths

summary = []

for pth, r in zip(sample_paths, results):
    # Count detections
    det_count = 0
    class_counts = {}
    if hasattr(r, "boxes") and r.boxes is not None:
        det_count = int(len(r.boxes))
        if det_count > 0 and hasattr(r.boxes, "cls"):
            cls = r.boxes.cls.detach().cpu().numpy().astype(int)
            for c in cls:
                class_counts[c] = class_counts.get(c, 0) + 1

    # Render + save annotated image
    plot_bgr = r.plot()
    plot_rgb = cv2.cvtColor(plot_bgr, cv2.COLOR_BGR2RGB)

    out_path = os.path.join(OUT_DIR, f"{Path(pth).stem}_pred.png")
    Image.fromarray(plot_rgb).save(out_path)

    summary.append((os.path.basename(pth), det_count, class_counts, out_path))

# ----------------------------
# Print summary
# ----------------------------
print("=== Summary (10 random images) ===")
for fname, det_count, class_counts, out_path in summary:
    print(f"- {fname:>14} | dets={det_count:3d} | class_counts={class_counts} | saved={out_path}")

print(f"\n✅ Saved annotated predictions to: {OUT_DIR}")

# ----------------------------
# Visualize grid (2x5)
# ----------------------------
plt.figure(figsize=(20, 8))
for i, (_, _, _, out_path) in enumerate(summary, start=1):
    img = Image.open(out_path).convert("RGB")
    plt.subplot(2, 5, i)
    plt.imshow(img)
    plt.title(f"#{i}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Create a dataset with patched images

In [ ]:
# ===========================
# CREATE ONE COMBINED PATCHED DATASET (450 imgs)
# ===========================
import os, glob, shutil
import torch
from PIL import Image
from torchvision import transforms
from torchvision.utils import save_image
from tqdm import tqdm

# ---- SETTINGS ----
PATCH_PATH   = "/content/drive/MyDrive/yolo_patch/billr.pt"
DATASET_ROOT = "/content/drive/MyDrive/2dod/2dod"     # contains billboard01..billboard09
OUT_ROOT     = "/content/drive/MyDrive/RANDOM/billr_patched_all"

SCENARIOS = [f"billboard{i:02d}" for i in range(1, 10)]
X_OFFSET, Y_OFFSET = 400, 270
IMGSZ = 640

# ---- OUTPUT DIRS (ONE dataset) ----
out_images_dir = os.path.join(OUT_ROOT, "images", "test")
out_labels_dir = os.path.join(OUT_ROOT, "labels", "test")
os.makedirs(out_images_dir, exist_ok=True)
os.makedirs(out_labels_dir, exist_ok=True)

# Optional: clean output first
# shutil.rmtree(OUT_ROOT); os.makedirs(out_images_dir, exist_ok=True); os.makedirs(out_labels_dir, exist_ok=True)

# ---- DEVICE ----
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- LOAD PATCH ----
patch = torch.load(PATCH_PATH, map_location="cpu")

# handle patch shapes like (1,3,H,W)
if isinstance(patch, (list, tuple)):
    patch = patch[0]
if patch.ndim == 4:
    patch = patch[0]
patch = patch.float()

# normalize if needed
if patch.max().item() > 1.0:
    patch = patch / 255.0

patch = patch.to(device).clamp(0, 1)
print("Loaded patch:", PATCH_PATH, "shape:", tuple(patch.shape), "device:", patch.device)

# ---- TRANSFORM ----
transform = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.ToTensor(),
])

def overlay_patch(image_tensor, patch_tensor, x_offset=X_OFFSET, y_offset=Y_OFFSET):
    patched = image_tensor.clone()
    _, H, W = image_tensor.shape
    _, ph, pw = patch_tensor.shape

    if x_offset < 0 or y_offset < 0 or x_offset + pw > W or y_offset + ph > H:
        raise ValueError(f"Patch out of bounds: img=({H},{W}) patch=({ph},{pw}) offset=({x_offset},{y_offset})")

    patched[:, y_offset:y_offset+ph, x_offset:x_offset+pw] = patch_tensor
    return patched

# ---- PROCESS ALL SCENARIOS INTO ONE FOLDER ----
exts = ("*.png", "*.jpg", "*.jpeg")
total = 0
per_scenario = {}

with torch.no_grad():
    for scenario in SCENARIOS:
        images_dir = os.path.join(DATASET_ROOT, scenario, "images", "test_nopatch")
        labels_dir = os.path.join(DATASET_ROOT, scenario, "labels", "test_nopatch")

        if not os.path.exists(images_dir):
            print(f"Skipping {scenario}: missing {images_dir}")
            continue

        img_paths = []
        for e in exts:
            img_paths += glob.glob(os.path.join(images_dir, e))
        img_paths = sorted(img_paths)

        if len(img_paths) == 0:
            print(f"Skipping {scenario}: no images found")
            continue

        count = 0
        for img_path in tqdm(img_paths, desc=f"Patching {scenario}", leave=False):
            img_name = os.path.basename(img_path)
            stem, ext = os.path.splitext(img_name)

            # Rename to avoid collisions when merging folders
            new_img_name = f"{scenario}_{stem}{ext}"
            new_label_name = f"{scenario}_{stem}.txt"

            img = Image.open(img_path).convert("RGB")
            img_tensor = transform(img).to(device)

            patched_tensor = overlay_patch(img_tensor, patch, X_OFFSET, Y_OFFSET)

            # Save patched image
            save_image(patched_tensor.detach().cpu(), os.path.join(out_images_dir, new_img_name))

            # Copy label if exists, else create empty label
            src_label = os.path.join(labels_dir, stem + ".txt")
            dst_label = os.path.join(out_labels_dir, new_label_name)
            if os.path.exists(src_label):
                shutil.copy(src_label, dst_label)
            else:
                open(dst_label, "w").close()

            total += 1
            count += 1

        per_scenario[scenario] = count

print("\n✅ Combined patched dataset created!")
print("Output root:", OUT_ROOT)
print("Images folder (should be ~450 imgs):", out_images_dir)
print("Labels folder:", out_labels_dir)
print("Total images saved:", total)
print("Per-scenario counts:", per_scenario)
print("Patch offsets used:", (X_OFFSET, Y_OFFSET))


In [ ]:
# ===========================
# EVALUATION CELL (combined dataset) - FIXED YAML (train key included)
# ===========================
!pip -q install ultralytics pyyaml tqdm

import os, glob
import yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

# ---------------------------
# SETTINGS
# ---------------------------
MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
DATA_ROOT  = "/content/drive/MyDrive/RANDOM/billr_patched_all"  # combined dataset root

CONFS = [0.001, 0.003]
IMGSZ = 640
NMS_IOU = 0.7
IOU_MATCH = 0.50  # IoU threshold to decide TP vs FP for FP/img metric

images_dir = os.path.join(DATA_ROOT, "images", "test")
labels_dir = os.path.join(DATA_ROOT, "labels", "test")
print("images_dir:", images_dir, "exists:", os.path.exists(images_dir))
print("labels_dir:", labels_dir, "exists:", os.path.exists(labels_dir))
if not os.path.exists(images_dir):
    raise FileNotFoundError(f"Missing images folder: {images_dir}")
if not os.path.exists(labels_dir):
    raise FileNotFoundError(f"Missing labels folder: {labels_dir}")

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)

# Try to include names (nice to have, not required)
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

# ---------------------------
# Temporary dataset YAML (Ultralytics requires train+val)
# We'll just point train/val/test all to the same folder for evaluation.
# ---------------------------
data_yaml_path = "/content/combined_billr_data.yaml"
d = {
    "path": DATA_ROOT,
    "train": "images/test",
    "val": "images/test",
    "test": "images/test",
}
if names_list is not None:
    d["names"] = names_list

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(d, f, sort_keys=False)

print("Wrote YAML:", data_yaml_path)

# ---------------------------
# Helpers for FP/img and Det/img
# ---------------------------
def yolo_label_to_xyxy(line, w, h):
    parts = line.strip().split()
    if len(parts) < 5:
        return None
    cls = int(float(parts[0]))
    xc, yc, bw, bh = map(float, parts[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def compute_fp_det_per_img(conf):
    exts = ("*.png", "*.jpg", "*.jpeg")
    img_paths = []
    for e in exts:
        img_paths.extend(glob.glob(os.path.join(images_dir, e)))
    img_paths = sorted(img_paths)
    if not img_paths:
        raise RuntimeError(f"No images found in {images_dir}")

    total_det, total_fp = 0, 0
    n_imgs = len(img_paths)

    with torch.no_grad():
        for p in tqdm(img_paths, desc=f"FP/Det @ conf={conf}", leave=False):
            img = Image.open(p).convert("RGB")
            w, h = img.size

            stem = os.path.splitext(os.path.basename(p))[0]
            lab_path = os.path.join(labels_dir, stem + ".txt")

            gts = []
            if os.path.exists(lab_path):
                with open(lab_path, "r") as f:
                    for line in f:
                        r = yolo_label_to_xyxy(line, w, h)
                        if r is not None:
                            gts.append(r)

            pred = model.predict(
                source=np.array(img),
                conf=conf,
                iou=NMS_IOU,   # NMS IoU
                imgsz=IMGSZ,
                verbose=False
            )
            boxes = pred[0].boxes
            if boxes is None or len(boxes) == 0:
                continue

            det_xyxy = boxes.xyxy.cpu().numpy()
            det_cls  = boxes.cls.cpu().numpy().astype(int)

            total_det += len(det_xyxy)

            # Greedy class-aware GT matching
            used_gt = set()
            for j in range(len(det_xyxy)):
                pj = det_xyxy[j]
                cj = det_cls[j]

                best_iou, best_k = 0.0, None
                for k, (cg, gg) in enumerate(gts):
                    if k in used_gt:
                        continue
                    if cg != cj:
                        continue
                    i = iou_xyxy(pj, gg)
                    if i > best_iou:
                        best_iou, best_k = i, k

                if best_k is not None and best_iou >= IOU_MATCH:
                    used_gt.add(best_k)  # TP
                else:
                    total_fp += 1        # FP

    return (total_fp / n_imgs), (total_det / n_imgs), n_imgs

# ---------------------------
# RUN EVAL
# ---------------------------
rows = []
for conf in CONFS:
    # mAP metrics
    val_res = model.val(data=data_yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
    map5095 = float(val_res.box.map)      # mAP50-95
    map50   = float(val_res.box.map50)    # mAP50

    # FP/img + Det/img
    fp_img, det_img, n_imgs = compute_fp_det_per_img(conf)

    rows.append({
        "conf": conf,
        "mAP50-95": map5095,
        "mAP50": map50,
        "FP/img": fp_img,
        "Det/img": det_img,
        "images": n_imgs
    })

    print(
        f"conf={conf:.3f} | "
        f"mAP50-95={map5095*100:.2f}% | mAP50={map50*100:.2f}% | "
        f"FP/img={fp_img:.4f} | Det/img={det_img:.4f} | imgs={n_imgs}"
    )

df = pd.DataFrame(rows)
print("\n=== SUMMARY TABLE ===")
display(df)

out_csv = os.path.join(DATA_ROOT, "evaluation_results_combined.csv")
df.to_csv(out_csv, index=False)
print("\n✅ Saved CSV to:", out_csv)


In [ ]:
# ===========================
# CELL 1: EVAL + PER-IMAGE COUNTS (FP and Det)
# ===========================
!pip -q install ultralytics pyyaml tqdm

import os, glob
import yaml
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

# -------- SETTINGS --------
MODEL_PATH = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
DATA_ROOT  = "/content/drive/MyDrive/RANDOM/billr_patched_all"

CONFS = [0.001, 0.3]   # <-- UPDATED
IMGSZ = 640
NMS_IOU = 0.7
IOU_MATCH = 0.50

images_dir = os.path.join(DATA_ROOT, "images", "test")
labels_dir = os.path.join(DATA_ROOT, "labels", "test")
if not os.path.exists(images_dir):
    raise FileNotFoundError(f"Missing images folder: {images_dir}")
if not os.path.exists(labels_dir):
    raise FileNotFoundError(f"Missing labels folder: {labels_dir}")

print("Using MODEL:", MODEL_PATH)
print("Using DATA_ROOT:", DATA_ROOT)
print("Images:", len(glob.glob(os.path.join(images_dir, '*'))))

model = YOLO(MODEL_PATH)

# optional names
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

# ---- Required YAML (train key needed by Ultralytics) ----
data_yaml_path = "/content/combined_billr_data.yaml"
d = {"path": DATA_ROOT, "train": "images/test", "val": "images/test", "test": "images/test"}
if names_list is not None:
    d["names"] = names_list
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(d, f, sort_keys=False)

def yolo_label_to_xyxy(line, w, h):
    parts = line.strip().split()
    if len(parts) < 5:
        return None
    cls = int(float(parts[0]))
    xc, yc, bw, bh = map(float, parts[1:5])
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def per_image_fp_det(conf):
    exts = ("*.png", "*.jpg", "*.jpeg")
    img_paths = []
    for e in exts:
        img_paths.extend(glob.glob(os.path.join(images_dir, e)))
    img_paths = sorted(img_paths)
    if not img_paths:
        raise RuntimeError(f"No images found in {images_dir}")

    fp_list = []
    det_list = []

    with torch.no_grad():
        for p in tqdm(img_paths, desc=f"Per-image FP/Det @ conf={conf}", leave=False):
            img = Image.open(p).convert("RGB")
            w, h = img.size

            stem = os.path.splitext(os.path.basename(p))[0]
            lab_path = os.path.join(labels_dir, stem + ".txt")

            gts = []
            if os.path.exists(lab_path):
                with open(lab_path, "r") as f:
                    for line in f:
                        r = yolo_label_to_xyxy(line, w, h)
                        if r is not None:
                            gts.append(r)

            pred = model.predict(
                source=np.array(img),
                conf=conf,
                iou=NMS_IOU,
                imgsz=IMGSZ,
                verbose=False
            )

            boxes = pred[0].boxes
            if boxes is None or len(boxes) == 0:
                fp_list.append(0)
                det_list.append(0)
                continue

            det_xyxy = boxes.xyxy.cpu().numpy()
            det_cls  = boxes.cls.cpu().numpy().astype(int)

            det_count = len(det_xyxy)
            fp_count = 0

            used_gt = set()
            for j in range(det_count):
                pj = det_xyxy[j]
                cj = det_cls[j]

                best_iou, best_k = 0.0, None
                for k, (cg, gg) in enumerate(gts):
                    if k in used_gt:
                        continue
                    if cg != cj:
                        continue
                    i = iou_xyxy(pj, gg)
                    if i > best_iou:
                        best_iou, best_k = i, k

                if best_k is not None and best_iou >= IOU_MATCH:
                    used_gt.add(best_k)  # TP
                else:
                    fp_count += 1        # FP

            fp_list.append(fp_count)
            det_list.append(det_count)

    return np.array(fp_list, dtype=float), np.array(det_list, dtype=float), len(img_paths)

# ---- Run Ultralytics mAP + per-image stats ----
eval_rows = []
per_image_stats = {}  # conf -> {"fp": arr, "det": arr, "n": n}

for conf in CONFS:
    val_res = model.val(data=data_yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
    map5095 = float(val_res.box.map)
    map50   = float(val_res.box.map50)

    fp_arr, det_arr, n_imgs = per_image_fp_det(conf)
    per_image_stats[conf] = {"fp": fp_arr, "det": det_arr, "n": n_imgs}

    eval_rows.append({"conf": conf, "mAP50-95": map5095, "mAP50": map50, "images": n_imgs})

    print(f"\nconf={conf:.3f} | mAP50-95={map5095*100:.2f}% | mAP50={map50*100:.2f}% | images={n_imgs}")


In [ ]:
# ===========================
# CELL 2: MEAN ± STD SUMMARY (FP/img, Det/img)
# ===========================
import numpy as np
import pandas as pd
import os

summary_rows = []

for conf, d in per_image_stats.items():
    fp = d["fp"]
    det = d["det"]
    n = d["n"]

    fp_mean  = float(fp.mean())
    fp_std   = float(fp.std(ddof=1)) if n > 1 else 0.0
    det_mean = float(det.mean())
    det_std  = float(det.std(ddof=1)) if n > 1 else 0.0

    m = [r for r in eval_rows if abs(r["conf"] - conf) < 1e-12][0]

    summary_rows.append({
        "conf": conf,
        "mAP50-95 (%)": m["mAP50-95"] * 100,
        "mAP50 (%)": m["mAP50"] * 100,
        "FP/img mean": fp_mean,
        "FP/img std": fp_std,
        "Det/img mean": det_mean,
        "Det/img std": det_std,
        "images": n
    })

df_summary = pd.DataFrame(summary_rows).sort_values("conf")
display(df_summary)

for _, r in df_summary.iterrows():
    print(
        f"conf={r['conf']:.3f} | "
        f"FP/img={r['FP/img mean']:.4f} ± {r['FP/img std']:.4f} | "
        f"Det/img={r['Det/img mean']:.4f} ± {r['Det/img std']:.4f} | "
        f"mAP50-95={r['mAP50-95 (%)']:.2f}% | mAP50={r['mAP50 (%)']:.2f}%"
    )

out_csv = os.path.join(DATA_ROOT, "evaluation_mean_std_summary.csv")
df_summary.to_csv(out_csv, index=False)
print("\n✅ Saved summary CSV to:", out_csv)


In [ ]:
# ===========================
# CELL 1: CREATE 9 PATCHED SCENARIO DATASETS (billboard01..09)
# ===========================
import os, glob, shutil
import torch
from PIL import Image
from torchvision import transforms
from torchvision.utils import save_image
from tqdm import tqdm

PATCH_PATH   = "/content/drive/MyDrive/yolo_patch/billr.pt"
DATASET_ROOT = "/content/drive/MyDrive/2dod/2dod"  # original dataset root
OUT_ROOT     = "/content/drive/MyDrive/RANDOM/billr_patched_scenarios"

SCENARIOS = [f"billboard{i:02d}" for i in range(1, 10)]
X_OFFSET, Y_OFFSET = 400, 270
IMGSZ = 640

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

patch = torch.load(PATCH_PATH, map_location="cpu")
if isinstance(patch, (list, tuple)):
    patch = patch[0]
if patch.ndim == 4:
    patch = patch[0]
patch = patch.float()
if patch.max().item() > 1.0:
    patch = patch / 255.0
patch = patch.to(device).clamp(0, 1)
print("Loaded patch:", PATCH_PATH, "shape:", tuple(patch.shape))

transform = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.ToTensor(),
])

def overlay_patch(image_tensor, patch_tensor, x_offset=X_OFFSET, y_offset=Y_OFFSET):
    patched = image_tensor.clone()
    _, H, W = image_tensor.shape
    _, ph, pw = patch_tensor.shape
    if x_offset < 0 or y_offset < 0 or x_offset + pw > W or y_offset + ph > H:
        raise ValueError(f"Patch out of bounds: img=({H},{W}) patch=({ph},{pw}) offset=({x_offset},{y_offset})")
    patched[:, y_offset:y_offset+ph, x_offset:x_offset+pw] = patch_tensor
    return patched

exts = ("*.png", "*.jpg", "*.jpeg")
total = 0

with torch.no_grad():
    for scenario in SCENARIOS:
        src_images = os.path.join(DATASET_ROOT, scenario, "images", "test_nopatch")
        src_labels = os.path.join(DATASET_ROOT, scenario, "labels", "test_nopatch")

        if not os.path.exists(src_images):
            print(f"Skipping {scenario}: missing {src_images}")
            continue

        out_images = os.path.join(OUT_ROOT, scenario, "images", "test")
        out_labels = os.path.join(OUT_ROOT, scenario, "labels", "test")
        os.makedirs(out_images, exist_ok=True)
        os.makedirs(out_labels, exist_ok=True)

        img_paths = []
        for e in exts:
            img_paths += glob.glob(os.path.join(src_images, e))
        img_paths = sorted(img_paths)

        if not img_paths:
            print(f"Skipping {scenario}: no images")
            continue

        for img_path in tqdm(img_paths, desc=f"Patching {scenario}", leave=False):
            img_name = os.path.basename(img_path)
            stem, _ = os.path.splitext(img_name)

            img = Image.open(img_path).convert("RGB")
            img_tensor = transform(img).to(device)
            patched_tensor = overlay_patch(img_tensor, patch)

            save_image(patched_tensor.detach().cpu(), os.path.join(out_images, img_name))

            src_lab = os.path.join(src_labels, stem + ".txt")
            dst_lab = os.path.join(out_labels, stem + ".txt")
            if os.path.exists(src_lab):
                shutil.copy(src_lab, dst_lab)
            else:
                open(dst_lab, "w").close()

            total += 1

print("\n✅ Done.")
print("Patched scenarios root:", OUT_ROOT)
print("Total patched images:", total)
print("Patch offsets used:", (X_OFFSET, Y_OFFSET))


In [ ]:
# ===========================
# CELL 2: EVAL EACH SCENARIO + MEAN ± STD ACROSS 9 SCENARIOS (ROBUST)
# ===========================
!pip -q install ultralytics pyyaml tqdm

import os, glob, re
import yaml
import numpy as np
import pandas as pd
from PIL import Image
import torch
from ultralytics import YOLO

# ---------------------------
# SETTINGS
# ---------------------------
MODEL_PATH   = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt"
PATCHED_ROOT = "/content/drive/MyDrive/RANDOM/billr_patched_scenarios"

SCENARIOS = [f"billboard{i:02d}" for i in range(1, 10)]
CONFS = [0.001, 0.3]     # your confs
IMGSZ = 640
NMS_IOU = 0.7            # NMS IoU for predict/val
IOU_MATCH = 0.50         # IoU threshold for matching preds->GT when counting FP

# ---------------------------
# Model
# ---------------------------
model = YOLO(MODEL_PATH)

# names for yaml (optional, but helpful)
names = getattr(model.model, "names", None) or getattr(model, "names", None)
if isinstance(names, dict):
    names_list = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    names_list = names
else:
    names_list = None

def write_yaml(root_dir, out_path):
    # Ultralytics requires train+val. We point them to the same folder for evaluation.
    d = {"path": root_dir, "train": "images/test", "val": "images/test", "test": "images/test"}
    if names_list is not None:
        d["names"] = names_list
    with open(out_path, "w") as f:
        yaml.safe_dump(d, f, sort_keys=False)

# ---------------------------
# Robust label parsing
# ---------------------------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def yolo_label_to_xyxy(line, w, h):
    """
    Robust YOLO label parser:
    extracts the first 5 numeric tokens: cls xc yc bw bh
    ignores empty/bad lines safely.
    """
    nums = _num_re.findall(line)
    if len(nums) < 5:
        return None
    cls = int(float(nums[0]))
    xc, yc, bw, bh = map(float, nums[1:5])

    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    return cls, np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def scenario_fp_det_means(images_dir, labels_dir, conf):
    exts = ("*.png", "*.jpg", "*.jpeg")
    img_paths = []
    for e in exts:
        img_paths.extend(glob.glob(os.path.join(images_dir, e)))
    img_paths = sorted(img_paths)
    if not img_paths:
        return None

    fp_list = []
    det_list = []

    with torch.no_grad():
        for p in img_paths:
            img = Image.open(p).convert("RGB")
            w, h = img.size
            stem = os.path.splitext(os.path.basename(p))[0]
            lab_path = os.path.join(labels_dir, stem + ".txt")

            gts = []
            if os.path.exists(lab_path):
                with open(lab_path, "r") as f:
                    for line in f:
                        r = yolo_label_to_xyxy(line, w, h)
                        if r is not None:
                            gts.append(r)

            pred = model.predict(
                source=np.array(img),
                conf=conf,
                iou=NMS_IOU,
                imgsz=IMGSZ,
                verbose=False
            )
            boxes = pred[0].boxes
            if boxes is None or len(boxes) == 0:
                fp_list.append(0)
                det_list.append(0)
                continue

            det_xyxy = boxes.xyxy.cpu().numpy()
            det_cls  = boxes.cls.cpu().numpy().astype(int)

            det_count = len(det_xyxy)
            fp_count = 0

            # Greedy, class-aware matching
            used_gt = set()
            for j in range(det_count):
                pj = det_xyxy[j]
                cj = det_cls[j]

                best_iou, best_k = 0.0, None
                for k, (cg, gg) in enumerate(gts):
                    if k in used_gt:
                        continue
                    if cg != cj:
                        continue
                    i = iou_xyxy(pj, gg)
                    if i > best_iou:
                        best_iou, best_k = i, k

                if best_k is not None and best_iou >= IOU_MATCH:
                    used_gt.add(best_k)  # TP
                else:
                    fp_count += 1        # FP

            fp_list.append(fp_count)
            det_list.append(det_count)

    fp_arr = np.array(fp_list, dtype=float)
    det_arr = np.array(det_list, dtype=float)
    return float(fp_arr.mean()), float(det_arr.mean()), len(img_paths)

# ---------------------------
# RUN: per-scenario metrics
# ---------------------------
rows = []

for scenario in SCENARIOS:
    root = os.path.join(PATCHED_ROOT, scenario)
    images_dir = os.path.join(root, "images", "test")
    labels_dir = os.path.join(root, "labels", "test")

    if not os.path.exists(images_dir):
        print("Skipping missing:", scenario)
        continue

    yaml_path = f"/content/{scenario}_billr.yaml"
    write_yaml(root, yaml_path)

    print(f"\n=== {scenario} ===")
    for conf in CONFS:
        # Ultralytics mAP
        val_res = model.val(data=yaml_path, conf=conf, iou=NMS_IOU, imgsz=IMGSZ, verbose=False)
        map5095 = float(val_res.box.map)
        map50   = float(val_res.box.map50)

        # FP/img and Det/img means for this scenario
        fp_mean, det_mean, n_imgs = scenario_fp_det_means(images_dir, labels_dir, conf)

        rows.append({
            "scenario": scenario,
            "conf": conf,
            "mAP50-95 (%)": map5095 * 100,
            "mAP50 (%)": map50 * 100,
            "FP/img mean": fp_mean,
            "Det/img mean": det_mean,
            "images": n_imgs
        })

df = pd.DataFrame(rows).sort_values(["conf", "scenario"])
print("\n=== PER-SCENARIO RESULTS ===")
display(df)

# ---------------------------
# Mean ± std ACROSS SCENARIOS
# (std computed over the 9 scenario-level values)
# ---------------------------
agg_rows = []
for conf in CONFS:
    dconf = df[df["conf"] == conf].copy()
    agg_rows.append({
        "conf": conf,
        "mAP50-95 mean": dconf["mAP50-95 (%)"].mean(),
        "mAP50-95 std":  dconf["mAP50-95 (%)"].std(ddof=1),
        "mAP50 mean":    dconf["mAP50 (%)"].mean(),
        "mAP50 std":     dconf["mAP50 (%)"].std(ddof=1),
        "FP/img mean":   dconf["FP/img mean"].mean(),
        "FP/img std":    dconf["FP/img mean"].std(ddof=1),
        "Det/img mean":  dconf["Det/img mean"].mean(),
        "Det/img std":   dconf["Det/img mean"].std(ddof=1),
        "scenarios": int(dconf["scenario"].nunique())
    })

df_agg = pd.DataFrame(agg_rows).sort_values("conf")
print("\n=== MEAN ± STD ACROSS 9 SCENARIOS ===")
display(df_agg)

for _, r in df_agg.iterrows():
    print(
        f"conf={r['conf']:.3f} | "
        f"mAP50-95={r['mAP50-95 mean']:.2f} ± {r['mAP50-95 std']:.2f} | "
        f"mAP50={r['mAP50 mean']:.2f} ± {r['mAP50 std']:.2f} | "
        f"FP/img={r['FP/img mean']:.4f} ± {r['FP/img std']:.4f} | "
        f"Det/img={r['Det/img mean']:.4f} ± {r['Det/img std']:.4f}"
    )

# ---------------------------
# Save CSVs
# ---------------------------
out1 = os.path.join(PATCHED_ROOT, "per_scenario_results.csv")
out2 = os.path.join(PATCHED_ROOT, "mean_std_across_scenarios.csv")
df.to_csv(out1, index=False)
df_agg.to_csv(out2, index=False)
print("\n✅ Saved:", out1)
print("✅ Saved:", out2)
